In [2]:
import json
import numpy as np
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

DATASET_FILES = {
    'CKD':          'results_CKD.json',
    'Parkinsons':   'results_Parkinsons.json',
    'Mammographic': 'results_Mammographic.json',
    'ILPD':         'results_ILPD.json',
    'SPECT':        'results_SPECT.json',
    'BreastCancer': 'results_BreastCancer.json',
}

METRICS = [
    'accuracy', 'roc_auc', 'f1', 'balanced_accuracy',
    'sensitivity', 'specificity', 'pr_auc'
]

COMPARISON_GROUPS = {
    'G1_ablation_components_sigmoid_h100': [
        'ELM-linear-h100',
        'ELM-sigmoid-h100',
        'ELM-SelfGate-sigmoid-h100',
        'ELM-DNM-noreg-sigmoid-h100',
        'ELM-DNM-sigmoid-h100',
    ],
    'G2_activation_effect_ELM_h100': [
        'ELM-linear-h100',
        'ELM-sigmoid-h100',
        'ELM-relu-h100',
        'ELM-tanh-h100',
    ],
    'G3_activation_effect_ELMDNM_h100': [
        'ELM-DNM-linear-h100',
        'ELM-DNM-sigmoid-h100',
        'ELM-DNM-relu-h100',
        'ELM-DNM-tanh-h100',
    ],
    'G4_hidden_size_effect_ELMDNM_sigmoid': [
        'ELM-DNM-sigmoid-h50',
        'ELM-DNM-sigmoid-h100',
        'ELM-DNM-sigmoid-h200',
    ],
    'G5_regularization_effect': [
        'ELM-DNM-noreg-sigmoid-h100',
        'ELM-DNM-noreg-relu-h100',
        'ELM-DNM-noreg-tanh-h100',
        'ELM-DNM-sigmoid-h100',
        'ELM-DNM-relu-h100',
        'ELM-DNM-tanh-h100',
    ],
    'G6_ELMDNM_vs_baselines_sigmoid_h100': [
        'ELM-DNM-sigmoid-h100',
        'ELM-sigmoid-h100',
        'DNM-sigmoid-h100',
        'MODN-sigmoid-h100',
        'RandomForest',
        'SVM',
        'MLP',
    ],
    'G7_ELMDNM_best_vs_all_baselines': [
        'ELM-DNM-sigmoid-h100',
        'ELM-DNM-relu-h100',
        'ELM-DNM-tanh-h100',
        'DNM-sigmoid-h100',
        'DNM-relu-h100',
        'MODN-sigmoid-h100',
        'MODN-relu-h100',
        'RandomForest',
        'SVM',
        'MLP',
    ],
}

COMPARISON_GROUPS_DESCRIPTION = {
    'G1_ablation_components_sigmoid_h100':   'Ablation: progressive component addition (sigmoid, h=100)',
    'G2_activation_effect_ELM_h100':         'Ablation: activation function effect on vanilla ELM (h=100)',
    'G3_activation_effect_ELMDNM_h100':      'Ablation: activation function effect on ELM-DNM (h=100)',
    'G4_hidden_size_effect_ELMDNM_sigmoid':  'Ablation: hidden size effect on ELM-DNM (sigmoid)',
    'G5_regularization_effect':              'Ablation: ridge regularization effect on ELM-DNM (sigmoid/relu/tanh)',
    'G6_ELMDNM_vs_baselines_sigmoid_h100':   'Main comparison: ELM-DNM vs all baselines (sigmoid, h=100)',
    'G7_ELMDNM_best_vs_all_baselines':       'Main comparison: ELM-DNM variants vs gradient and sklearn baselines',
}


def cohens_dz(scores_a, scores_b):
    diff     = np.array(scores_a, dtype=float) - np.array(scores_b, dtype=float)
    std_diff = np.std(diff, ddof=1)
    return float(np.mean(diff) / std_diff) if std_diff > 0.0 else 0.0


def effect_size_label(d):
    abs_d = abs(d)
    if abs_d < 0.2:  return 'negligible'
    if abs_d < 0.5:  return 'small'
    if abs_d < 0.8:  return 'medium'
    return 'large'


def run_group_tests(group_name, model_names, fold_values_dict, metric):
    available = [m for m in model_names if m in fold_values_dict]
    if len(available) < 2:
        return None

    score_matrix = np.array([fold_values_dict[m] for m in available])

    if score_matrix.shape[1] < 3:
        return None

    try:
        friedman_stat, friedman_p = friedmanchisquare(*score_matrix)
    except Exception:
        return None

    model_pairs  = list(combinations(range(len(available)), 2))
    raw_p_values = []
    effect_sizes = []

    for i, j in model_pairs:
        try:
            _, p_w = wilcoxon(score_matrix[i], score_matrix[j])
        except ValueError:
            p_w = 1.0
        raw_p_values.append(float(p_w))
        effect_sizes.append(cohens_dz(score_matrix[i], score_matrix[j]))

    reject_flags, holm_p_values, _, _ = multipletests(raw_p_values, method='holm')

    pairwise_results = [
        {
            'model_a':        available[i],
            'model_b':        available[j],
            'mean_a':         float(np.mean(score_matrix[i])),
            'mean_b':         float(np.mean(score_matrix[j])),
            'mean_diff':      float(np.mean(score_matrix[i]) - np.mean(score_matrix[j])),
            'p_wilcoxon':     raw_p_values[k],
            'p_holm':         float(holm_p_values[k]),
            'reject_h0':      bool(reject_flags[k]),
            'cohens_dz':      effect_sizes[k],
            'effect_label':   effect_size_label(effect_sizes[k]),
        }
        for k, (i, j) in enumerate(model_pairs)
    ]

    significant_pairs = [p for p in pairwise_results if p['reject_h0']]

    return {
        'models_tested':        available,
        'n_models':             len(available),
        'n_pairs':              len(model_pairs),
        'friedman_statistic':   float(friedman_stat),
        'friedman_p_value':     float(friedman_p),
        'friedman_significant': bool(friedman_p < 0.05),
        'n_significant_pairs':  len(significant_pairs),
        'pairwise_results':     pairwise_results,
        'significant_pairs':    significant_pairs,
    }


for dataset_name, filepath in DATASET_FILES.items():
    print(f'\n{"=" * 70}')
    print(f'DATASET: {dataset_name}')
    print(f'{"=" * 70}')

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f'  [SKIP] File not found: {filepath}')
        continue

    summary_data       = data.get('summary', {})
    available_models   = list(summary_data.keys())
    dataset_results    = {}

    for metric in METRICS:
        fold_values_dict = {}
        for model_name in available_models:
            values = summary_data.get(model_name, {}).get(metric, {}).get('values', [])
            if values and len(values) >= 3:
                finite_values = [v for v in values if v is not None and not (isinstance(v, float) and np.isnan(v))]
                if len(finite_values) >= 3:
                    fold_values_dict[model_name] = finite_values

        metric_results = {}
        for group_name, model_list in COMPARISON_GROUPS.items():
            result = run_group_tests(group_name, model_list, fold_values_dict, metric)
            if result is not None:
                metric_results[group_name] = result

        dataset_results[metric] = metric_results

        print(f'\n  Metric: {metric.upper()}')
        for group_name, result in metric_results.items():
            desc = COMPARISON_GROUPS_DESCRIPTION.get(group_name, group_name)
            sig  = 'SIGNIFICANT' if result['friedman_significant'] else 'not significant'
            print(f'    [{group_name}] {desc}')
            print(f'      Friedman chi2={result["friedman_statistic"]:.4f}  p={result["friedman_p_value"]:.6f}  ({sig})')
            print(f'      Models tested : {result["n_models"]}  Pairs: {result["n_pairs"]}  Significant pairs (Holm): {result["n_significant_pairs"]}')
            if result['significant_pairs']:
                for pair in result['significant_pairs']:
                    print(f'        > {pair["model_a"]:42s} vs {pair["model_b"]:42s}  '
                          f'p_holm={pair["p_holm"]:.4f}  d={pair["cohens_dz"]:+.3f} ({pair["effect_label"]})  '
                          f'diff={pair["mean_diff"]:+.4f}')

print(f'\n{"=" * 70}')
print(f'Datasets processed : {list(DATASET_FILES.keys())}')
print(f'Metrics tested     : {METRICS}')
print(f'Groups tested      : {list(COMPARISON_GROUPS.keys())}')




DATASET: CKD

  Metric: ACCURACY
    [G1_ablation_components_sigmoid_h100] Ablation: progressive component addition (sigmoid, h=100)
      Friedman chi2=26.7442  p=0.000022  (SIGNIFICANT)
      Models tested : 5  Pairs: 10  Significant pairs (Holm): 4
        > ELM-linear-h100                            vs ELM-sigmoid-h100                            p_holm=0.0195  d=-4.128 (large)  diff=-0.2700
        > ELM-linear-h100                            vs ELM-SelfGate-sigmoid-h100                   p_holm=0.0195  d=-3.824 (large)  diff=-0.3050
        > ELM-linear-h100                            vs ELM-DNM-noreg-sigmoid-h100                  p_holm=0.0195  d=-3.496 (large)  diff=-0.3050
        > ELM-linear-h100                            vs ELM-DNM-sigmoid-h100                        p_holm=0.0195  d=-3.496 (large)  diff=-0.3050
    [G2_activation_effect_ELM_h100] Ablation: activation function effect on vanilla ELM (h=100)
      Friedman chi2=25.5312  p=0.000012  (SIGNIFICANT)
      Models